In [ ]:
import pandas as pd
import numpy as np

print("Loading Ta Feng data...")

df = pd.read_csv('../data/raw/ta_feng_all_months_merged.csv')


df = df[['CUSTOMER_ID', 'TRANSACTION_DT', 'PRODUCT_ID']]


df = df.drop_duplicates()

print(f"Total unique transactions (baskets): {df.groupby(['CUSTOMER_ID', 'TRANSACTION_DT']).ngroups}")
print(f"Total unique users: {df['CUSTOMER_ID'].nunique()}")
print(f"Total unique items: {df['PRODUCT_ID'].nunique()}")

Since Ta Feng's dataset contains over 23,000 (23, 812 to be exact) unique items, most of which rarely appear
Truncate the dataset for top 1,000 items

In [ ]:
print("Calculating item frequencies...")
item_counts = df['PRODUCT_ID'].value_counts()


TOP_K = 1000
top_items = set(item_counts.head(TOP_K).index)

print(f"Top {TOP_K} items capture {item_counts.head(TOP_K).sum() / len(df):.2%} of all item interactions.")


print("Mapping rare items...")
df['ITEM_MAPPED'] = df['PRODUCT_ID'].apply(lambda x: x if x in top_items else 0)


df['TRANSACTION_DT'] = pd.to_datetime(df['TRANSACTION_DT'])
df = df.sort_values(by=['CUSTOMER_ID', 'TRANSACTION_DT'])


print("Data mapped and sorted chronologically.")

Transform the raw product id into contiguous indicies(0 indexed) for the DL models to understand

In [ ]:
unique_items = [0] + list(top_items) 
item_to_idx = {item: idx for idx, item in enumerate(unique_items)}


print("Mapping raw IDs to contiguous indices (0-1000)...")
df['ITEM_IDX'] = df['ITEM_MAPPED'].map(item_to_idx)

Create customer baskets by grouping the data of one customer present in multiple rows for the same day into a single list of items

In [ ]:
print("Grouping items into individual baskets...")

baskets = df.groupby(['CUSTOMER_ID', 'TRANSACTION_DT'])['ITEM_IDX'].apply(list).reset_index()


baskets = baskets.sort_values(['CUSTOMER_ID', 'TRANSACTION_DT'])

print(f"Total baskets formed: {len(baskets)}")
print(baskets.head())

Group individual baskets based on cutomer_id
We need atleast three baskets per user; 2 for training and 1 for target prediction

In [ ]:
print("Grouping baskets into user sequences...")

user_sequences = baskets.groupby('CUSTOMER_ID')['ITEM_IDX'].apply(list).reset_index()
user_sequences.rename(columns={'ITEM_IDX': 'BASKET_SEQUENCE'}, inplace=True)


min_baskets = 3
user_sequences = user_sequences[user_sequences['BASKET_SEQUENCE'].apply(len) >= min_baskets]

print(f"Users retained (>= {min_baskets} baskets): {len(user_sequences)}")


sample_seq = user_sequences.iloc[0]['BASKET_SEQUENCE']
print(f"\nSample user sequence length (number of trips): {len(sample_seq)}")
print(f"Trip 1 items: {sample_seq[0]}")
print(f"Trip 2 items: {sample_seq[1]}")